# Gold Pipeline Health Mart

**Purpose**: ETL operational metrics for monitoring pipeline health and data quality.

**Target Table**: `workspace.gold.gold_pipeline_health`

**Grain**: One row per pipeline run per date

**Key Metrics**:
- Rows read and written
- Runtime performance
- Success/failure rates
- Data quality indicators

**Data Sources**:
- `workspace.warehouse.fact_pipeline_runs`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_pipeline_health
USING DELTA
COMMENT 'Pipeline operational metrics for monitoring and alerting'
AS

WITH pipeline_metrics AS (
  SELECT
    fpr.run_date_sk AS health_date_sk,
    fpr.pipeline_name,
    fpr.source_name,
    fpr.batch_id,
    fpr.status,
    
    -- MEASURES: Volume metrics
    fpr.rows_read,
    fpr.rows_written,
    fpr.runtime_seconds,
    
    -- DERIVED: Data quality metrics
    CASE 
      WHEN fpr.rows_read > 0 
      THEN CAST(fpr.rows_written AS DECIMAL(10,4)) / fpr.rows_read
      ELSE NULL 
    END AS write_ratio,
    
    -- DERIVED: Performance metrics
    CASE 
      WHEN fpr.runtime_seconds > 0 
      THEN CAST(fpr.rows_written AS DECIMAL(18,2)) / fpr.runtime_seconds
      ELSE NULL 
    END AS rows_per_second
    
  FROM workspace.warehouse.fact_pipeline_runs fpr
  WHERE fpr.run_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 90), 'yyyyMMdd') AS INT)
),

daily_aggregates AS (
  SELECT
    pm.health_date_sk,
    pm.pipeline_name,
    pm.source_name,
    
    -- MEASURES: Aggregated metrics
    COUNT(*) AS run_count,
    COUNT(CASE WHEN pm.status = 'SUCCESS' THEN 1 END) AS success_count,
    COUNT(CASE WHEN pm.status = 'FAILURE' THEN 1 END) AS failure_count,
    COUNT(CASE WHEN pm.status = 'TIMEOUT' THEN 1 END) AS timeout_count,
    
    SUM(pm.rows_read) AS total_rows_read,
    SUM(pm.rows_written) AS total_rows_written,
    SUM(pm.runtime_seconds) AS total_runtime_seconds,
    
    ROUND(AVG(pm.runtime_seconds), 2) AS avg_runtime_seconds,
    MAX(pm.runtime_seconds) AS max_runtime_seconds,
    MIN(pm.runtime_seconds) AS min_runtime_seconds,
    
    ROUND(AVG(pm.write_ratio), 4) AS avg_write_ratio,
    ROUND(AVG(pm.rows_per_second), 2) AS avg_rows_per_second
    
  FROM pipeline_metrics pm
  GROUP BY pm.health_date_sk, pm.pipeline_name, pm.source_name
),

temporal_enriched AS (
  SELECT
    da.*,
    
    -- TEMPORAL METRICS: Prior period comparisons
    LAG(da.total_rows_written, 1) OVER (
      PARTITION BY da.pipeline_name, da.source_name
      ORDER BY da.health_date_sk
    ) AS prev_day_rows_written,
    
    LAG(da.avg_runtime_seconds, 1) OVER (
      PARTITION BY da.pipeline_name, da.source_name
      ORDER BY da.health_date_sk
    ) AS prev_day_avg_runtime,
    
    -- 7-day rolling averages
    AVG(da.total_rows_written) OVER (
      PARTITION BY da.pipeline_name, da.source_name
      ORDER BY da.health_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_rows_written,
    
    AVG(da.avg_runtime_seconds) OVER (
      PARTITION BY da.pipeline_name, da.source_name
      ORDER BY da.health_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_runtime
    
  FROM daily_aggregates da
)

SELECT
  -- DIMENSIONS
  te.health_date_sk,
  te.pipeline_name,
  te.source_name,
  
  -- MEASURES: Run counts
  te.run_count,
  te.success_count,
  te.failure_count,
  te.timeout_count,
  
  -- DERIVED: Success rate
  CASE 
    WHEN te.run_count > 0 
    THEN CAST(te.success_count AS DECIMAL(10,4)) / te.run_count
    ELSE NULL 
  END AS success_rate,
  
  -- MEASURES: Volume metrics
  te.total_rows_read,
  te.total_rows_written,
  te.total_runtime_seconds,
  
  -- MEASURES: Performance metrics
  te.avg_runtime_seconds,
  te.max_runtime_seconds,
  te.min_runtime_seconds,
  te.avg_write_ratio,
  te.avg_rows_per_second,
  
  -- TEMPORAL METRICS
  te.rolling_7day_avg_rows_written,
  te.rolling_7day_avg_runtime,
  
  -- DERIVED: Performance changes
  CASE 
    WHEN te.prev_day_rows_written > 0 
    THEN CAST((te.total_rows_written - te.prev_day_rows_written) AS DECIMAL(10,4)) / te.prev_day_rows_written
    ELSE NULL 
  END AS pct_change_rows_vs_prev_day,
  
  CASE 
    WHEN te.prev_day_avg_runtime > 0 
    THEN CAST((te.avg_runtime_seconds - te.prev_day_avg_runtime) AS DECIMAL(10,4)) / te.prev_day_avg_runtime
    ELSE NULL 
  END AS pct_change_runtime_vs_prev_day,
  
  -- DERIVED: Anomaly indicators
  CASE 
    WHEN te.rolling_7day_avg_rows_written > 0 
      AND ABS(te.total_rows_written - te.rolling_7day_avg_rows_written) / te.rolling_7day_avg_rows_written > 0.5
    THEN TRUE
    ELSE FALSE
  END AS volume_anomaly_flag,
  
  CASE 
    WHEN te.rolling_7day_avg_runtime > 0 
      AND te.avg_runtime_seconds > te.rolling_7day_avg_runtime * 1.5
    THEN TRUE
    ELSE FALSE
  END AS performance_anomaly_flag,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
ORDER BY te.health_date_sk DESC, te.pipeline_name;

In [0]:
%sql
-- Validation: Pipeline health summary
SELECT
  pipeline_name,
  source_name,
  COUNT(*) AS total_days,
  SUM(run_count) AS total_runs,
  ROUND(AVG(success_rate), 4) AS avg_success_rate,
  SUM(total_rows_written) AS total_rows_written,
  ROUND(AVG(avg_runtime_seconds), 2) AS avg_runtime_seconds,
  SUM(CASE WHEN volume_anomaly_flag = TRUE THEN 1 ELSE 0 END) AS volume_anomaly_days,
  SUM(CASE WHEN performance_anomaly_flag = TRUE THEN 1 ELSE 0 END) AS perf_anomaly_days
FROM workspace.gold.gold_pipeline_health
GROUP BY pipeline_name, source_name
ORDER BY total_rows_written DESC;